[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvvvvivekkk/AI-Engineer/blob/main/07-llms-and-genai/notebooks/02-calling-llm-apis.ipynb)

# 02 - Calling LLM APIs: OpenAI, Anthropic and Gemini

**Goal:** learn the request shape of the three big provider SDKs — **messages and roles**, **system prompts**, **streaming**, **structured JSON output**, **errors and retries** — in a way that runs **with or without API keys**.

- If a key is set (`OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GEMINI_API_KEY`), the real API is called.
- If not, every call uses a clearly labelled **offline fallback**: a tiny local model (SmolLM2-135M) or a fake response. You still see the real code.

**Never paste a key into a notebook.** Put it in a `.env` file (see `.env.example` in this module) or, on Colab, in the Secrets panel (key icon on the left).

## Setup
Install the three official SDKs (they are small) plus `python-dotenv`, which loads keys from a `.env` file.

In [1]:
import importlib.util, subprocess, sys
# Install a package only if it is missing (Colab already has most of these).
def ensure(pkg, module=None):
    if importlib.util.find_spec(module or pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
for pkg, mod in [("openai", "openai"), ("anthropic", "anthropic"), ("google-genai", "google.genai"),
                 ("python-dotenv", "dotenv"), ("pydantic", "pydantic"), ("transformers", "transformers")]:
    ensure(pkg, mod)
import openai, anthropic
from google import genai
print("openai", openai.__version__, "| anthropic", anthropic.__version__, "| google-genai", genai.__version__)

openai 3.57.0 | anthropic 2.12.0 | google-genai 3.6.1


## 1. Load keys safely

`load_dotenv()` copies `KEY=value` lines from a `.env` file into **environment variables**. We print only *whether* each key exists — never the key itself. On Colab, the commented lines show how to read from Secrets.

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()          # looks for a .env file in this folder or its parents

# On Colab, uncomment to copy secrets into environment variables:
# from google.colab import userdata
# for k in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GEMINI_API_KEY"]:
#     try: os.environ[k] = userdata.get(k)
#     except Exception: pass

HAS = {
    "openai": bool(os.getenv("OPENAI_API_KEY")),
    "anthropic": bool(os.getenv("ANTHROPIC_API_KEY")),
    "gemini": bool(os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")),
}
# Example model names (late 2026) - they change often; override in .env. Check each provider's models page.
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
ANTHROPIC_MODEL = os.getenv("ANTHROPIC_MODEL", "claude-haiku-5-5")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-flash-latest")
print("Keys found:", HAS)

Keys found: {'openai': False, 'anthropic': False, 'gemini': False}


## 2. The offline fallback
This function answers with SmolLM2 on the CPU. It takes the same `system` + `messages` as the real helpers and returns the same kind of dictionary, so the rest of the notebook does not care which one ran.

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

LOCAL_MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"
_tok = AutoTokenizer.from_pretrained(LOCAL_MODEL)
_model = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL, dtype=torch.float32)   # float32: fast on CPU

def local_chat(system, messages, max_new_tokens=80):
    """OFFLINE FALLBACK: answer with a tiny local model. Same inputs/outputs as the API helpers below."""
    chat = [{"role": "system", "content": system}] + list(messages)
    inputs = _tok.apply_chat_template(chat, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    with torch.no_grad():
        out = _model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    n_in = inputs["input_ids"].shape[1]
    return {"text": _tok.decode(out[0][n_in:], skip_special_tokens=True).strip(),
            "input_tokens": n_in, "output_tokens": out.shape[1] - n_in,
            "source": "offline-fallback (SmolLM2-135M)"}

print(local_chat("Answer in one sentence.", [{"role": "user", "content": "What is Python?"}]))

/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:  57%|█████▋    | 154/272 [00:00<00:00, 1536.33it/s]

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 1582.78it/s]

{'text': 'Python is a high-level, interpreted programming language developed by Guido van Rossom. It is widely used for web development, data analysis, machine learning, and automation.', 'input_tokens': 23, 'output_tokens': 36, 'source': 'offline-fallback (SmolLM2-135M)'}


## 3. Messages and roles

A chat request is a list of messages. Roles: **system** (rules for the assistant), **user** (the person), **assistant** (the model's earlier replies). Each provider puts the system prompt in a slightly different place:

| Provider | System prompt goes in | Assistant role name |
|---|---|---|
| OpenAI (Responses API) | `instructions=` | `assistant` |
| Anthropic | `system=` | `assistant` |
| Gemini | `config=GenerateContentConfig(system_instruction=...)` | `model` |

In [4]:
SYSTEM = "You are a concise tutor for beginner programmers. Answer in at most two sentences."
QUESTION = [{"role": "user", "content": "What is an API key and why must I keep it secret?"}]
print(SYSTEM); print(QUESTION)

You are a concise tutor for beginner programmers. Answer in at most two sentences.
[{'role': 'user', 'content': 'What is an API key and why must I keep it secret?'}]


## 4. OpenAI

The **Responses API** (`client.responses.create`) is OpenAI's current main endpoint; `resp.output_text` joins the text output and `resp.usage` holds token counts. The real call runs only if a key exists.

In [5]:
from openai import OpenAI

def ask_openai(system, messages, max_tokens=200):
    if not HAS["openai"]:
        return local_chat(system, messages)                 # offline fallback
    client = OpenAI()                                       # reads OPENAI_API_KEY
    resp = client.responses.create(model=OPENAI_MODEL, instructions=system, input=messages,
                                   max_output_tokens=max_tokens)
    return {"text": resp.output_text, "input_tokens": resp.usage.input_tokens,
            "output_tokens": resp.usage.output_tokens, "source": f"openai:{OPENAI_MODEL}"}

print(ask_openai(SYSTEM, QUESTION))

{'text': 'An API key is a unique identifier used by a software application to authenticate and authorize access to its services. It is a secret value that must be kept secret to prevent unauthorized access. It is used to authenticate users and authorize access to APIs, which are often used for data exchange and data processing.', 'input_tokens': 43, 'output_tokens': 60, 'source': 'offline-fallback (SmolLM2-135M)'}


## 5. Anthropic (Claude)

`client.messages.create` needs `max_tokens`. The reply's `content` is a list of **blocks**; text blocks have `.text`. `stop_reason` says why it stopped (`end_turn`, `max_tokens`, ...).

In [6]:
from anthropic import Anthropic

def ask_anthropic(system, messages, max_tokens=200):
    if not HAS["anthropic"]:
        return local_chat(system, messages)
    client = Anthropic()                                    # reads ANTHROPIC_API_KEY
    msg = client.messages.create(model=ANTHROPIC_MODEL, max_tokens=max_tokens, system=system, messages=messages)
    text = "".join(b.text for b in msg.content if b.type == "text")
    return {"text": text, "input_tokens": msg.usage.input_tokens,
            "output_tokens": msg.usage.output_tokens, "source": f"anthropic:{ANTHROPIC_MODEL}"}

print(ask_anthropic(SYSTEM, QUESTION))

{'text': 'An API key is a unique identifier used by a software application to authenticate and authorize access to its services. It is a secret value that must be kept secret to prevent unauthorized access. It is used to authenticate users and authorize access to APIs, which are often used for data exchange and data processing.', 'input_tokens': 43, 'output_tokens': 60, 'source': 'offline-fallback (SmolLM2-135M)'}


## 6. Google Gemini

The `google-genai` SDK uses `client.models.generate_content`. History is a list of `Content` objects with `parts`; the assistant role is called `model`. Keep the client in a variable (writing `genai.Client().models...` in one line can fail because Python may close the unnamed client early).

In [7]:
from google.genai import types

def ask_gemini(system, messages, max_tokens=200):
    if not HAS["gemini"]:
        return local_chat(system, messages)
    client = genai.Client()                                 # reads GEMINI_API_KEY / GOOGLE_API_KEY
    contents = [types.Content(role="model" if m["role"] == "assistant" else "user",
                              parts=[types.Part(text=m["content"])]) for m in messages]
    resp = client.models.generate_content(model=GEMINI_MODEL, contents=contents,
        config=types.GenerateContentConfig(system_instruction=system, max_output_tokens=max_tokens))
    u = resp.usage_metadata
    return {"text": resp.text, "input_tokens": u.prompt_token_count,
            "output_tokens": u.candidates_token_count, "source": f"gemini:{GEMINI_MODEL}"}

print(ask_gemini(SYSTEM, QUESTION))

{'text': 'An API key is a unique identifier used by a software application to authenticate and authorize access to its services. It is a secret value that must be kept secret to prevent unauthorized access. It is used to authenticate users and authorize access to APIs, which are often used for data exchange and data processing.', 'input_tokens': 43, 'output_tokens': 60, 'source': 'offline-fallback (SmolLM2-135M)'}


## 7. Same question, three providers
Put the answers side by side in a table. Without keys all three rows come from the fallback, but the code path is identical.

In [8]:
import pandas as pd
rows = []
for name, fn in [("openai", ask_openai), ("anthropic", ask_anthropic), ("gemini", ask_gemini)]:
    r = fn(SYSTEM, QUESTION)
    rows.append({"provider": name, "source": r["source"], "in": r["input_tokens"], "out": r["output_tokens"],
                 "answer": r["text"][:90]})
pd.set_option("display.max_colwidth", 100)
pd.DataFrame(rows)

,provider,source,in,out,answer
0,openai,offline-fallback (SmolLM2-135M),43,60,An API key is a unique identifier used by a software application to authenticate and autho
1,anthropic,offline-fallback (SmolLM2-135M),43,60,An API key is a unique identifier used by a software application to authenticate and autho
2,gemini,offline-fallback (SmolLM2-135M),43,60,An API key is a unique identifier used by a software application to authenticate and autho


## 8. The API has no memory

Each call is independent. To continue a conversation you must send the **whole history**. Compare asking "What is my name?" with and without the earlier turns.

In [9]:
no_history = [{"role": "user", "content": "What is my name?"}]
with_history = [
    {"role": "user", "content": "Hi, my name is Asha."},
    {"role": "assistant", "content": "Hello Asha! How can I help?"},
    {"role": "user", "content": "What is my name?"},
]
print("WITHOUT history:", ask_anthropic("Answer briefly.", no_history)["text"])
print("WITH history:   ", ask_anthropic("Answer briefly.", with_history)["text"])

WITHOUT history: My name is Emily Wilson.


WITH history:    My name is Asha.


## 9. Streaming

**Streaming** shows text as it is generated. Real code for each provider is below (it runs only with a key). The offline fallback streams locally with Hugging Face's `TextIteratorStreamer`, which runs generation in a background **thread** (a second line of work inside the same program) and hands us text pieces as they appear.

In [10]:
from threading import Thread
from transformers import TextIteratorStreamer

def stream_local(system, messages):
    chat = [{"role": "system", "content": system}] + messages
    inputs = _tok.apply_chat_template(chat, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    streamer = TextIteratorStreamer(_tok, skip_prompt=True, skip_special_tokens=True)
    Thread(target=_model.generate, kwargs={**inputs, "streamer": streamer, "max_new_tokens": 60}).start()
    for piece in streamer:
        print(piece, end="", flush=True)
    print()

def stream_answer(provider, system, messages):
    if provider == "openai" and HAS["openai"]:
        client = OpenAI()
        for event in client.responses.create(model=OPENAI_MODEL, instructions=system, input=messages, stream=True):
            if event.type == "response.output_text.delta":
                print(event.delta, end="", flush=True)
        print()
    elif provider == "anthropic" and HAS["anthropic"]:
        client = Anthropic()
        with client.messages.stream(model=ANTHROPIC_MODEL, max_tokens=200, system=system, messages=messages) as s:
            for text in s.text_stream:
                print(text, end="", flush=True)
        print()
    elif provider == "gemini" and HAS["gemini"]:
        client = genai.Client()
        for chunk in client.models.generate_content_stream(model=GEMINI_MODEL, contents=messages[-1]["content"],
                config=types.GenerateContentConfig(system_instruction=system)):
            print(chunk.text or "", end="", flush=True)
        print()
    else:
        print("[offline fallback, streamed locally] ", end="")
        stream_local(system, messages)

stream_answer("anthropic", "You are a poet.", [{"role": "user", "content": "Write two short lines about the monsoon."}])

[offline fallback, streamed locally] 

The 

monsoon's 

a 

cruel 

and 

... (38 outputs left out)

soothing 

serenade. 

It's 

a

## 10. Structured JSON output

Programs need data in a fixed shape. We describe the shape with a **Pydantic** model (a Python class that validates data). Each SDK has a helper that sends the **schema** and parses the reply:

- OpenAI: `client.responses.parse(..., text_format=Ticket).output_parsed`
- Anthropic: `client.messages.parse(..., output_format=Ticket).parsed_output`
- Gemini: `config=GenerateContentConfig(response_mime_type="application/json", response_schema=Ticket)` then `.parsed`

In [11]:
from pydantic import BaseModel, ValidationError

class Ticket(BaseModel):
    category: str     # billing / delivery / technical / other
    urgency: int      # 1 (low) .. 5 (high)
    summary: str

MESSAGE = "My order #4521 has not arrived after 10 days and nobody answers the phone!"

def classify(provider):
    instr = "Classify the customer message into category, urgency (1-5) and a one-line summary."
    if provider == "openai" and HAS["openai"]:
        client = OpenAI()
        return client.responses.parse(model=OPENAI_MODEL, instructions=instr, input=MESSAGE,
                                      text_format=Ticket).output_parsed
    if provider == "anthropic" and HAS["anthropic"]:
        client = Anthropic()
        return client.messages.parse(model=ANTHROPIC_MODEL, max_tokens=300, system=instr,
                                     messages=[{"role": "user", "content": MESSAGE}],
                                     output_format=Ticket).parsed_output
    if provider == "gemini" and HAS["gemini"]:
        client = genai.Client()
        return client.models.generate_content(model=GEMINI_MODEL, contents=f"{instr}\n\n{MESSAGE}",
            config=types.GenerateContentConfig(response_mime_type="application/json",
                                               response_schema=Ticket)).parsed
    return None   # no key for this provider

for p in ["openai", "anthropic", "gemini"]:
    print(p, "->", classify(p) or "(no key: skipped)")

openai -> (no key: skipped)
anthropic -> (no key: skipped)
gemini -> (no key: skipped)


### Validation matters: what if the model returns bad JSON?

Without native structured output (older or tiny models), you ask for JSON in the prompt and **must validate**. Our tiny local model often gets the format wrong — a perfect demonstration. We try to parse; if it fails we fall back to a safe default instead of crashing.

In [12]:
import json, re

def classify_local(message):
    prompt = ('Reply with ONLY a JSON object with keys "category" (string), "urgency" (integer 1-5) '
              f'and "summary" (string) for this customer message: {message}')
    raw = local_chat("You output JSON only.", [{"role": "user", "content": prompt}], max_new_tokens=60)["text"]
    print("raw model output:", repr(raw[:150]))
    try:
        match = re.search(r"\{.*\}", raw, re.S)              # grab the first {...} block, if any
        return Ticket(**json.loads(match.group(0))), "parsed"
    except (AttributeError, json.JSONDecodeError, ValidationError, TypeError) as e:
        print("validation failed:", type(e).__name__)
        return Ticket(category="other", urgency=3, summary=message[:60]), "safe default"

ticket, how = classify_local(MESSAGE)
print(how, "->", ticket)

raw model output: 'Here is the JSON object with keys "category", "urgency", "summary", and "phone":\n\n```json\n{\n  "category": "urgency",\n  "urgency": 1,\n  "summary": "My '
validation failed: AttributeError
safe default -> category='other' urgency=3 summary='My order #4521 has not arrived after 10 days and nobody answ'


## 11. Errors, status codes and retries

Real calls fail sometimes. We can trigger a **real** SDK error safely, with no key and no internet: point each client at a closed port on this computer (`127.0.0.1:9`). The SDK builds a genuine request, fails to connect, retries according to its settings, and raises its typed exception. This also proves our call syntax is accepted by the installed SDKs.

In [13]:
import time
BAD = "http://127.0.0.1:9"     # nothing listens here -> connection refused, instantly

t0 = time.time()
try:
    OpenAI(api_key="not-a-real-key", base_url=BAD + "/v1", max_retries=2).responses.create(
        model=OPENAI_MODEL, instructions=SYSTEM, input=QUESTION, max_output_tokens=50)
except openai.APIConnectionError as e:
    print(f"OpenAI    -> {type(e).__name__} after {time.time()-t0:.1f}s (includes 2 automatic retries)")

try:
    Anthropic(api_key="not-a-real-key", base_url=BAD, max_retries=0).messages.create(
        model=ANTHROPIC_MODEL, max_tokens=50, system=SYSTEM, messages=QUESTION)
except anthropic.APIConnectionError as e:
    print("Anthropic ->", type(e).__name__)

try:
    gclient = genai.Client(api_key="not-a-real-key", http_options=types.HttpOptions(
        base_url=BAD, retry_options=types.HttpRetryOptions(attempts=1)))
    gclient.models.generate_content(model=GEMINI_MODEL, contents="hi")
except Exception as e:
    print("Gemini    ->", type(e).__name__)

OpenAI    -> APIConnectionError after 1.4s (includes 2 automatic retries)
Anthropic -> APIConnectionError
Gemini    -> ConnectError


The pattern for real code: catch the **most specific** errors first. Only temporary errors (429 rate limit, 5xx server errors, 529 overloaded, connection problems) are worth retrying; 400/401/403/404 mean "fix your request or key".

| Exception (OpenAI / Anthropic) | HTTP code | Retry? |
|---|---|---|
| `BadRequestError` | 400 | no |
| `AuthenticationError` | 401 | no |
| `PermissionDeniedError` | 403 | no |
| `NotFoundError` | 404 (often a wrong model name) | no |
| `RateLimitError` | 429 | yes, after waiting |
| `InternalServerError` | 5xx | yes |
| `APIConnectionError` / `APITimeoutError` | none | yes |

Here is a safe wrapper that tries a provider and falls back to the local model on any API failure:

In [14]:
def safe_ask(provider_fn, system, messages):
    try:
        return provider_fn(system, messages)
    except (openai.RateLimitError, anthropic.RateLimitError):
        print("Rate limited: slow down, then retry later. Using fallback for now.")
    except (openai.AuthenticationError, anthropic.AuthenticationError):
        print("Bad API key: check your .env file. Using fallback.")
    except (openai.APIStatusError, anthropic.APIStatusError) as e:
        print(f"HTTP error {e.status_code}: {e.message}. Using fallback.")
    except (openai.APIConnectionError, anthropic.APIConnectionError):
        print("Network problem. Using fallback.")
    return local_chat(system, messages)

print(safe_ask(ask_anthropic, SYSTEM, QUESTION)["source"])

offline-fallback (SmolLM2-135M)


### Exponential backoff

The OpenAI and Anthropic SDKs already retry temporary errors (2 times by default; set `max_retries=` on the client). When you need your own loop — for logging, or to switch models — use **exponential backoff with jitter**: wait 1s, 2s, 4s ... plus a little randomness. Below, a fake API fails 60% of the time so you can watch it work (waits are shortened for the demo).

In [15]:
import random
random.seed(1)

class TemporaryError(Exception): pass

def flaky_api(prompt):
    if random.random() < 0.6:
        raise TemporaryError("429 rate limit (simulated)")
    return f"answer to {prompt!r}"

def with_backoff(fn, *args, attempts=5, base=0.1):
    for i in range(attempts):
        try:
            return fn(*args)
        except TemporaryError as e:
            if i == attempts - 1:
                raise
            wait = base * 2 ** i + random.uniform(0, base)
            print(f"  attempt {i+1} failed ({e}); retrying in {wait:.2f}s")
            time.sleep(wait)

print(with_backoff(flaky_api, "What is top-p?"))

  attempt 1 failed (429 rate limit (simulated)); retrying in 0.18s
answer to 'What is top-p?'


## Try it yourself
1. Add a fourth helper `ask_local` and include it in the comparison table.
2. Change `SYSTEM` to make the tutor answer like a pirate. Does the fallback model follow it?
3. Add a `language: str` field to `Ticket`. Update `classify_local` and check validation.
4. If you have a free Gemini key, put it in `.env` and rerun the notebook. Which cells change?